# Visualizing Text Data

Text is data, but it does not arrive in a neat table of quantitative and categorical features. Before we can visualize text, we need to decide what information to extract from it.

In this lecture, we will use movie reviews to ask:

> What words and patterns distinguish positive reviews from negative reviews?

We will move from raw review text to word frequencies, comparative visualizations, quantitative text features, and finally an optional similarity analysis.



## 1. Load and inspect the reviews

The original Stanford Large Movie Review Dataset contains 50,000 labeled reviews. The local teaching file is a balanced 10,000-review sample created from that dataset so the examples remain responsive for students.



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/imdb_reviews_sample.csv')
df.head()



In [ ]:
df.info()



In [ ]:
df['sentiment_label'] = df['sentiment'].map({'pos': 'Positive', 'neg': 'Negative'})

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='sentiment_label', order=['Negative', 'Positive'])
plt.title('Number of Reviews by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.show()



The reviews are evenly divided between positive and negative examples. The `review` column contains the text we will visualize, while `sentiment` supplies a label that lets us compare groups.



## 2. Prepare the text

Text preprocessing is not neutral: the choices we make determine what will appear in the visualizations. Here we will lowercase the reviews, keep alphabetic words, remove English stopwords, and remove a few movie-specific words that occur in nearly every review.



In [ ]:
import re
import math
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

additional_stopwords = {'movie', 'movies', 'film', 'films'}
stop_words = set(ENGLISH_STOP_WORDS) | additional_stopwords

def tokenize(text):
    words = re.findall(r"[a-zA-Z]+", str(text).lower())
    return [word for word in words if len(word) > 2 and word not in stop_words]

df['tokens'] = df['review'].map(tokenize)
df['clean_text'] = df['tokens'].str.join(' ')
df['review_length_words'] = df['tokens'].str.len()

df[['sentiment_label', 'review', 'clean_text', 'review_length_words']].head()



## 3. Start with a word cloud

Word clouds are useful as a first visual summary because they make prominent words easy to notice. They are less useful for precise comparisons, so we will follow them with charts that encode frequency more directly.



In [ ]:
from wordcloud import WordCloud

def show_wordcloud(sentiment, title):
    text = ' '.join(df.loc[df['sentiment'] == sentiment, 'clean_text'])
    wordcloud = WordCloud(width=900, height=450, background_color='white').generate(text)
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.show()

show_wordcloud('pos', 'Words in Positive Movie Reviews')
show_wordcloud('neg', 'Words in Negative Movie Reviews')



What can we learn from these word clouds? What can they hide? Remember that word size represents frequency, not importance, and that word clouds do not show word order or context.



## 4. Compare the most frequent words



In [ ]:
from collections import Counter

def frequency_frame(sentiment, number_of_words=15):
    counts = Counter(
        word
        for tokens in df.loc[df['sentiment'] == sentiment, 'tokens']
        for word in tokens
    )
    return pd.DataFrame(counts.most_common(number_of_words), columns=['word', 'frequency'])

positive_frequency = frequency_frame('pos')
negative_frequency = frequency_frame('neg')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(data=negative_frequency, x='frequency', y='word', ax=axes[0], color='#d95f02')
axes[0].set_title('Most Frequent Words in Negative Reviews')
axes[0].set_xlabel('Frequency')
axes[0].set_ylabel('Word')
sns.barplot(data=positive_frequency, x='frequency', y='word', ax=axes[1], color='#1b9e77')
axes[1].set_title('Most Frequent Words in Positive Reviews')
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('Word')
plt.tight_layout()
plt.show()



The two bar charts make exact frequencies easier to compare than the word clouds. However, words that occur frequently in both groups may still tell us little about the difference between positive and negative reviews.



## 5. Find words that distinguish the two groups

A document-frequency comparison asks a more focused question: in what proportion of positive or negative reviews does a word appear? Requiring a minimum number of reviews helps prevent proper names that occur in only a few reviews from dominating the chart.



In [ ]:
positive_counts = Counter(
    word for tokens in df.loc[df['sentiment'] == 'pos', 'tokens'] for word in set(tokens)
)
negative_counts = Counter(
    word for tokens in df.loc[df['sentiment'] == 'neg', 'tokens'] for word in set(tokens)
)

vocabulary = {
    word for word in set(positive_counts) | set(negative_counts)
    if positive_counts[word] + negative_counts[word] >= 100
}
positive_total = (df['sentiment'] == 'pos').sum()
negative_total = (df['sentiment'] == 'neg').sum()

comparison = pd.DataFrame([
    {
        'word': word,
        'log2_ratio': math.log2(
            (positive_counts[word] + 1) / (positive_total + 2)
        ) - math.log2(
            (negative_counts[word] + 1) / (negative_total + 2)
        )
    }
    for word in vocabulary
])

comparison['magnitude'] = comparison['log2_ratio'].abs()
comparison = comparison.sort_values('magnitude', ascending=False).head(20).sort_values('log2_ratio')

colors = ['#d95f02' if value < 0 else '#1b9e77' for value in comparison['log2_ratio']]
plt.figure(figsize=(10, 7))
plt.barh(comparison['word'], comparison['log2_ratio'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Words That Distinguish Negative and Positive Reviews')
plt.xlabel('Log2 Document-Frequency Ratio: Positive minus Negative')
plt.ylabel('Word')
plt.show()



Words to the right are relatively more common in positive reviews. Words to the left are relatively more common in negative reviews. This chart is more analytical than a word cloud, but it still does not prove that a word causes a sentiment.



## 6. Turn reviews into a quantitative feature

A text document can also be summarized by its length. Do positive and negative reviews tend to contain different numbers of words?



In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df,
    x='sentiment_label',
    y='review_length_words',
    order=['Negative', 'Positive']
)
plt.title('Review Length by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Number of Words')
plt.show()



Review length is a simple text feature. It may reveal differences between groups, but it is not a complete representation of what a review means.



## 7. Compare a calculated sentiment score with the label

The dataset supplies a positive or negative label. We can also calculate a sentiment polarity score with TextBlob and examine where the two approaches agree or disagree.



In [ ]:
#%pip install textblob
from textblob import TextBlob

df['textblob_polarity'] = df['review'].map(lambda text: TextBlob(text).sentiment.polarity)

plt.figure(figsize=(8, 5))
sns.violinplot(
    data=df,
    x='sentiment_label',
    y='textblob_polarity',
    order=['Negative', 'Positive']
)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('TextBlob Polarity by Dataset Sentiment Label')
plt.xlabel('Dataset Sentiment')
plt.ylabel('TextBlob Polarity')
plt.show()



TextBlob is a rule-based estimate, not a ground-truth measurement. Reviews with sarcasm, mixed opinions, or complex context may receive a surprising score.



## Optional final section: cosine similarity

Cosine similarity compares documents after they have been converted into vectors. Here we will compare a small set of actual movie reviews rather than unrelated toy sentences.

Reviews with similar word patterns should have larger similarity values. The values are not sentiment scores: two negative reviews can be similar to each other, and a positive and negative review can still share many words.



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

selected_reviews = pd.concat([
    df[df['sentiment'] == 'neg'].sample(4, random_state=42),
    df[df['sentiment'] == 'pos'].sample(4, random_state=42)
]).reset_index(drop=True)

selected_reviews['document'] = [
    'Negative review ' + str(number) for number in range(1, 5)
] + [
    'Positive review ' + str(number) for number in range(1, 5)
]

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = vectorizer.fit_transform(selected_reviews['review'])
similarity_matrix = cosine_similarity(tfidf_matrix)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=selected_reviews['document'],
    columns=selected_reviews['document']
)

plt.figure(figsize=(9, 7))
sns.heatmap(similarity_df, annot=True, cmap='Blues', vmin=0, vmax=1, fmt='.2f')
plt.title('Cosine Similarity Between Selected Movie Reviews')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()



## Discussion

- Which visualization made the largest difference to your understanding of the reviews?
- What did the word clouds show that the frequency charts did not?
- Which words distinguished the sentiment groups most strongly?
- What limitations should we keep in mind when turning text into numerical features?
- Which pair of reviews had the highest cosine similarity, and why might they be similar?
